In [ ]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch_geometric.data import Data, Batch, Dataset as PyGDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import subgraph, to_dense_batch
from torch_scatter import scatter_add, scatter_mean

from rdkit import Chem
from rdkit.Chem import AllChem, rdchem
from sklearn.metrics import balanced_accuracy_score, accuracy_score

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ==============================================================================
# 0. CONFIGURATION
# ==============================================================================
CONFIG = {
    # --- Analysis Settings ---
    "NUM_CONFORMERS": 20,     # Ensemble size per ligand
    "SAMPLE_SIZE": 2000,      # Number of pairs to test
    "SEED": 42,
    "DEVICE": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    
    # --- Paths (Adjust for Github structure) ---
    # Note: Users need the test predictions CSV and the trained model checkpoint
    "PREDICTIONS_PATH": Path("../results/predictions_test.csv"), 
    "LIGAND_DB_PATH": Path("../data/raw/GPCRactDB_v1.csv"), # Needs SMILES column
    "PROTEIN_GRAPH_DIR": Path("../Data/Protein_Graphs_PyG/"),
    "MODEL_SAVE_PATH": Path("../models/saved/gpcract_final.pt"),
    "OUTPUT_DIR": Path("./results/sensitivity_analysis/"),
    
    # --- Model Hyperparameters (Must match training) ---
    "HIDDEN_DIM": 128,
    "PROTEIN_LAYERS": 4,
    "PROPAGATION_ATTENTION_LAYERS": 3,
    "ATTENTION_HEADS": 4,
    "PROTEIN_TYPE": "gated_residual",
    "LIGAND_TYPE": "gated_residual",
    "ELEMENT_EMBEDDING_DIM": 8,
    "DROPOUT": 0.0, # Inference mode
}

CONFIG['OUTPUT_DIR'].mkdir(parents=True, exist_ok=True)

In [ ]:
# ==============================================================================
# 1. MODEL ARCHITECTURE
# ==============================================================================

def unsorted_segment_sum(data, segment_ids, num_segments):
    out = data.new_zeros((num_segments, data.size(1)))
    scatter_add(data, segment_ids, out=out, dim=0)
    return out

class E_GCL_Gated(nn.Module):
    def __init__(self, input_nf, output_nf, hidden_nf, edges_in_d=0, act_fn=nn.SiLU(), residual=True, attention=False, normalize=False, coords_agg='mean', tanh=False):
        super(E_GCL_Gated, self).__init__()
        input_edge = input_nf * 2
        self.residual = residual
        self.attention = attention
        self.normalize = normalize
        self.coords_agg = coords_agg
        self.tanh = tanh
        self.epsilon = 1e-8
        edge_coords_nf = 1

        self.edge_mlp = nn.Sequential(
            nn.Linear(input_edge + edge_coords_nf + edges_in_d, hidden_nf),
            act_fn,
            nn.Linear(hidden_nf, hidden_nf),
            act_fn)
        
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_nf + input_nf, hidden_nf),
            act_fn,
            nn.Linear(hidden_nf, output_nf))
        
        self.gate_mlp = nn.Sequential(
            nn.Linear(hidden_nf + input_nf, hidden_nf),
            act_fn,
            nn.Linear(hidden_nf, output_nf),
            nn.Sigmoid())

        self.node_norm = nn.LayerNorm(output_nf)
        
        layer = nn.Linear(hidden_nf, 1, bias=False)
        torch.nn.init.xavier_uniform_(layer.weight, gain=0.001)
        coord_mlp_list = [nn.Linear(hidden_nf, hidden_nf), act_fn, layer]
        if self.tanh: coord_mlp_list.append(nn.Tanh())
        self.coord_mlp = nn.Sequential(*coord_mlp_list)

        if self.attention:
            self.att_mlp = nn.Sequential(nn.Linear(hidden_nf, 1), nn.Sigmoid())

    def coord2radial(self, edge_index, coord):
        row, col = edge_index
        coord_diff = coord[row] - coord[col]
        radial = torch.sum(coord_diff**2, dim=1, keepdim=True)
        if self.normalize:
            norm = torch.sqrt(radial + self.epsilon)
            coord_diff = coord_diff / norm
        return radial, coord_diff

    def edge_model(self, h_row, h_col, radial, edge_attr):
        out = torch.cat([h_row, h_col, radial] + ([edge_attr] if edge_attr is not None else []), dim=1)
        out = self.edge_mlp(out)
        if self.attention:
            out = out * self.att_mlp(out)
        return out

    def node_model(self, x, edge_index, edge_feat, node_attr=None):
        row, col = edge_index
        agg = unsorted_segment_sum(edge_feat, row, num_segments=x.size(0))
        agg_cat = torch.cat([x, agg] + ([node_attr] if node_attr is not None else []), dim=1)
        update_val = self.node_mlp(agg_cat)
        gate_val = self.gate_mlp(agg_cat)
        out = x * gate_val + (x + update_val) * (1 - gate_val) if self.residual else x * gate_val + update_val * (1 - gate_val)
        return self.node_norm(out)

    def coord_model(self, coord, edge_index, coord_diff, edge_feat):
        row, col = edge_index
        trans = coord_diff * self.coord_mlp(edge_feat)
        if self.coords_agg == 'sum':
            agg = unsorted_segment_sum(trans, row, num_segments=coord.size(0))
        elif self.coords_agg == 'mean':
            agg = unsorted_segment_sum(trans, row, num_segments=coord.size(0)) / (unsorted_segment_sum(torch.ones_like(trans), row, num_segments=coord.size(0)) + 1e-8)
        else:
            raise Exception('Wrong coords_agg parameter')
        return coord + agg

    def forward(self, h, edge_index, coord, edge_attr=None, node_attr=None):
        row, col = edge_index
        radial, coord_diff = self.coord2radial(edge_index, coord)
        e_ij = self.edge_model(h[row], h[col], radial, edge_attr)
        coord = self.coord_model(coord, edge_index, coord_diff, e_ij)
        h = self.node_model(h, edge_index, e_ij, node_attr)
        return h, coord, e_ij

class EGNN_Gated_GlobalResidual(nn.Module):
    def __init__(self, in_node_nf, hidden_nf, out_node_nf, in_edge_nf=0, n_layers=4, residual=True, attention=False, normalize=False, coords_agg='mean', tanh=False):
        super().__init__()
        self.hidden_nf = hidden_nf
        self.n_layers = n_layers
        self.embedding_in = nn.Linear(in_node_nf, hidden_nf)
        self.embedding_out = nn.Linear(hidden_nf, out_node_nf)
        for i in range(n_layers):
            self.add_module(f"gcl_{i}", E_GCL_Gated(hidden_nf, hidden_nf, hidden_nf, edges_in_d=in_edge_nf, residual=residual, attention=attention, normalize=normalize, coords_agg=coords_agg, tanh=tanh))

    def forward(self, h, coord, edge_index, edge_attr=None):
        device = self.embedding_in.weight.device
        h, coord, edge_index = h.to(device), coord.to(device), edge_index.to(device)
        if edge_attr is not None: edge_attr = edge_attr.to(device)
        
        h_initial = self.embedding_in(h)
        h = h_initial
        for i in range(self.n_layers):
            h, coord, _ = self._modules[f"gcl_{i}"](h, edge_index, coord, edge_attr=edge_attr)
        h = h + h_initial
        return self.embedding_out(h), coord

class GPCRact_Model(nn.Module):
    """
    Cleaned GPCRact Model for Sensitivity Analysis.
    Removed class/family embeddings.
    """
    def __init__(self, protein_in_dim_clean, protein_in_dim_full, ligand_in_dim, hidden_dim,
                 protein_config, ligand_config, element_embedding_dim,
                 n_attn_heads, dropout, propagation_attention_layers):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # --- Module 1: Interaction ---
        self.element_embedding = nn.Embedding(num_embeddings=6, embedding_dim=element_embedding_dim)
        # Encoders
        self.bs_encoder = EGNN_Gated_GlobalResidual(protein_in_dim_clean, hidden_dim, hidden_dim, n_layers=protein_config['n_layers'], attention=True, tanh=True)
        self.ligand_encoder = EGNN_Gated_GlobalResidual(ligand_in_dim, hidden_dim, hidden_dim, n_layers=ligand_config['n_layers'], attention=True, tanh=True)
        
        self.p_to_l_attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=n_attn_heads, dropout=dropout, batch_first=True)
        self.l_to_p_attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=n_attn_heads, dropout=dropout, batch_first=True)

        # --- Module 2: Local Propagation ---
        self.local_propagation_encoder = EGNN_Gated_GlobalResidual(hidden_dim, hidden_dim, hidden_dim, n_layers=protein_config['n_layers'], attention=True, tanh=True)
        self.protein_embedding_ca = nn.Linear(protein_in_dim_full, hidden_dim)
        self.protein_embedding_sc = nn.Linear(protein_in_dim_full, hidden_dim)

        # --- Module 3: Global Propagation ---
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=n_attn_heads, dim_feedforward=hidden_dim * 4, dropout=dropout, batch_first=True)
        self.global_integration_transformer = nn.TransformerEncoder(encoder_layer, num_layers=propagation_attention_layers)
        self.final_norm = nn.LayerNorm(hidden_dim)

        # --- Module 4: Heads ---
        self.binding_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))
        self.activity_type_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 2))

    def forward(self, protein_batch, ligand_batch):
        # 1. Interaction
        bs_mask = protein_batch.bs_mask
        bs_edge_index, _ = subgraph(bs_mask, protein_batch.edge_index, relabel_nodes=True, num_nodes=protein_batch.num_nodes)
        
        p_features_bs = torch.cat([protein_batch.x_float_clean[bs_mask], self.element_embedding(protein_batch.x_elem[bs_mask])], dim=1)
        h_p_bs, _ = self.bs_encoder(p_features_bs, protein_batch.pos[bs_mask], bs_edge_index)
        h_l, _ = self.ligand_encoder(ligand_batch.x, ligand_batch.pos, ligand_batch.edge_index)

        padded_bs, mask_bs = to_dense_batch(h_p_bs, protein_batch.batch[bs_mask])
        padded_l, mask_l = to_dense_batch(h_l, ligand_batch.batch)

        # Cross Attention
        p_updated, _ = self.p_to_l_attention(query=padded_bs, key=padded_l, value=padded_l, key_padding_mask=~mask_l)
        l_updated, _ = self.l_to_p_attention(query=padded_l, key=padded_bs, value=padded_bs, key_padding_mask=~mask_bs)
        p_updated[~mask_bs] = 0
        l_updated[~mask_l] = 0

        # Pooling & Binding Prediction
        p_vec = p_updated.sum(dim=1) / mask_bs.sum(dim=1, keepdim=True).clamp(min=1)
        l_vec = l_updated.sum(dim=1) / mask_l.sum(dim=1, keepdim=True).clamp(min=1)
        binding_logit = self.binding_head(torch.cat([p_vec, l_vec], dim=1))

        # 2. Propagation (Gating)
        ligand_signal = p_updated[mask_bs]
        gate_weight = torch.sigmoid(binding_logit).squeeze(-1)[protein_batch.batch].unsqueeze(-1)
        gated_signal = ligand_signal * gate_weight[bs_mask]

        # Local EGNN
        p_features_full = torch.cat([protein_batch.x_float_full, self.element_embedding(protein_batch.x_elem)], dim=1)
        h = torch.zeros(p_features_full.size(0), self.hidden_dim, device=p_features_full.device)
        ca_mask, sc_mask = (protein_batch.node_roles == 0), (protein_batch.node_roles == 1)
        h[ca_mask] = self.protein_embedding_ca(p_features_full[ca_mask]).to(h.dtype)
        h[sc_mask] = self.protein_embedding_sc(p_features_full[sc_mask]).to(h.dtype)

        h_init = h.clone()
        coords = protein_batch.pos.clone()
        
        for i in range(self.local_propagation_encoder.n_layers):
            h[bs_mask] = h[bs_mask] + gated_signal
            h, coords, _ = self.local_propagation_encoder._modules[f"gcl_{i}"](h, protein_batch.edge_index, coords)
        
        h = self.final_norm(self.local_propagation_encoder.embedding_out(h + h_init))

        # 3. Global Integration
        padded_h, mask_p = to_dense_batch(h, protein_batch.batch)
        final_h = self.global_integration_transformer(padded_h, src_key_padding_mask=~mask_p)[mask_p]

        # 4. Activity Prediction
        # Note: Class/Family embeddings are removed here
        pool_p = scatter_mean(final_h, protein_batch.batch, dim=0)
        activity_logit = self.activity_type_head(torch.cat([pool_p, l_vec], dim=1))
        
        return binding_logit, activity_logit, coords

In [ ]:
# ==============================================================================
# 2. HELPER FUNCTIONS (RDKit & Graph)
# ==============================================================================

def get_atom_features(atom):
    possible_elements = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br', 'I', 'H']
    element = atom.GetSymbol()
    features = [int(element == s) for s in possible_elements] + [int(element not in possible_elements)]
    features.append(atom.GetDegree())
    features.append(atom.GetFormalCharge())
    hybridizations = [rdchem.HybridizationType.SP, rdchem.HybridizationType.SP2, rdchem.HybridizationType.SP3]
    features += [int(atom.GetHybridization() == h) for h in hybridizations]
    features.append(int(atom.GetIsAromatic()))
    features.append(atom.GetTotalNumHs())
    features.append(int(atom.IsInRing()))
    return features

def get_bond_features(bond):
    bond_types = [rdchem.BondType.SINGLE, rdchem.BondType.DOUBLE, rdchem.BondType.TRIPLE, rdchem.BondType.AROMATIC]
    features = [int(bond.GetBondType() == t) for t in bond_types]
    features.append(int(bond.GetIsConjugated()))
    features.append(int(bond.IsInRing()))
    return features

def smiles_to_ensemble_graphs(smiles, num_confs, seed):
    """Generates N conformers, removes hydrogens, returns PyG Data list."""
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None
    
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv2()
    params.randomSeed = seed
    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=num_confs, params=params)
    if not conf_ids: return None

    try:
        mol = Chem.RemoveHs(mol) # Remove Hs to match training schema
    except: return None

    atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
    edge_indices, edge_features = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_indices.extend([(i, j), (j, i)])
        bf = get_bond_features(bond)
        edge_features.extend([bf, bf])
    
    x_tensor = torch.tensor(atom_features, dtype=torch.float32)
    edge_index_tensor = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    edge_attr_tensor = torch.tensor(edge_features, dtype=torch.float32)

    graphs = []
    for i in range(mol.GetNumConformers()):
        pos = mol.GetConformer(i).GetPositions()
        data = Data(x=x_tensor, pos=torch.tensor(pos, dtype=torch.float32),
                    edge_index=edge_index_tensor, edge_attr=edge_attr_tensor, conf_id=i)
        graphs.append(data)
    return graphs

def preprocess_protein_graph(protein_graph):
    """Feature slicing logic (Same as training)"""
    p = protein_graph.clone()
    x = p.x
    
    h_res = x[:, :20]; h_bs = x[:, 20:21]; h_disp = x[:, 21:23]
    p.x_elem = x[:, 23].long()
    h_rel = x[:, 24:27]; h_dist = x[:, 27:28]; h_rdkit = x[:, 28:]
    
    p.x_float_full = torch.cat([h_res, h_bs, h_disp, h_rel, h_dist, h_rdkit], dim=1)
    p.x_float_clean = torch.cat([h_res, h_rel, h_dist, h_rdkit], dim=1)
    p.node_roles = p.node_role
    del p.x, p.node_role
    
    return p

In [ ]:
# ==============================================================================
# 3. INFERENCE & SENSITIVITY ANALYSIS
# ==============================================================================

def run_sensitivity_analysis():
    print("--- [Step 1] Loading Data & Sampling ---")
    try:
        pred_df = pd.read_csv(CONFIG['PREDICTIONS_PATH'])
        ligand_db = pd.read_csv(CONFIG['LIGAND_DB_PATH'])
        ikey_to_smiles = dict(zip(ligand_db.Ikey, ligand_db.SMILES))
    except Exception as e:
        print(f"Data load error: {e}. Please ensure paths are correct.")
        return

    # Target: True Binders predicted as Binders
    target_df = pred_df[(pred_df['Binding_Label'] == 1.0) & (pred_df['Binding_Prob'] > 0.5)].copy()
    
    if len(target_df) > CONFIG['SAMPLE_SIZE']:
        target_df = target_df.sample(n=CONFIG['SAMPLE_SIZE'], random_state=CONFIG['SEED'])
    
    target_df['SMILES'] = target_df['Ikey'].map(ikey_to_smiles)
    target_df = target_df.dropna(subset=['SMILES'])
    print(f"Analyzing {len(target_df)} ligand-receptor pairs.")

    # Load Model
    print("--- [Step 2] Loading Model ---")
    # Dummy load for dimensions
    sample_p = torch.load(list(CONFIG['PROTEIN_GRAPH_DIR'].glob("*.pt"))[0], map_location='cpu')
    sample_p = preprocess_protein_graph(sample_p)
    
    model = GPCRact_Model(
        protein_in_dim_clean=sample_p.x_float_clean.shape[1] + CONFIG['ELEMENT_EMBEDDING_DIM'],
        protein_in_dim_full=sample_p.x_float_full.shape[1] + CONFIG['ELEMENT_EMBEDDING_DIM'],
        ligand_in_dim=19,
        hidden_dim=CONFIG['HIDDEN_DIM'],
        protein_config={"type": CONFIG['PROTEIN_TYPE'], "n_layers": CONFIG['PROTEIN_LAYERS']},
        ligand_config={"type": CONFIG['LIGAND_TYPE'], "n_layers": CONFIG['LIGAND_LAYERS']},
        element_embedding_dim=CONFIG['ELEMENT_EMBEDDING_DIM'],
        dropout=0.0,
        n_attn_heads=CONFIG['ATTENTION_HEADS'],
        propagation_attention_layers=CONFIG['PROPAGATION_ATTENTION_LAYERS']
    )
    
    # Load state dict, ignoring missing keys (class embeddings)
    state = torch.load(CONFIG['MODEL_SAVE_PATH'], map_location=CONFIG['DEVICE'])
    state = {k:v for k,v in state.items() if "gpcr_class" not in k and "gpcr_family" not in k}
    model.load_state_dict(state, strict=False)
    model.to(CONFIG['DEVICE'])
    model.eval()

    # Run Inference
    print("--- [Step 3] Running Ensemble Inference ---")
    results = []
    p_cache = {}

    for _, row in tqdm(target_df.iterrows(), total=len(target_df)):
        ikey, uniprot, smiles = row['Ikey'], row['UniProt'], row['SMILES']
        
        # Generate Conformers
        l_graphs = smiles_to_ensemble_graphs(smiles, CONFIG['NUM_CONFORMERS'], CONFIG['SEED'])
        if not l_graphs: continue

        # Load Protein
        if uniprot not in p_cache:
            try:
                p_data = torch.load(CONFIG['PROTEIN_GRAPH_DIR'] / f"{uniprot}.pt", map_location='cpu')
                p_cache[uniprot] = preprocess_protein_graph(p_data)
            except: continue
        
        # Batching (Replicate protein N times)
        p_batch = Batch.from_data_list([p_cache[uniprot]] * len(l_graphs)).to(CONFIG['DEVICE'])
        l_batch = Batch.from_data_list(l_graphs).to(CONFIG['DEVICE'])

        with torch.no_grad():
            b_logit, a_logits, _ = model(p_batch, l_batch)
            b_probs = torch.sigmoid(b_logit).cpu().numpy().flatten()
            a_probs = torch.softmax(a_logits, dim=1).cpu().numpy()

        for i in range(len(l_graphs)):
            results.append({
                "Ikey": ikey, "UniProt": uniprot, "Conf_ID": i,
                "Binding_Prob": b_probs[i],
                "Prob_Antagonist": a_probs[i, 0],
                "Prob_Agonist": a_probs[i, 1],
                "Pred_Class": np.argmax(a_probs[i])
            })

    # Save
    df_res = pd.DataFrame(results)
    df_res.to_csv(CONFIG['OUTPUT_DIR'] / "ensemble_results.csv", index=False)
    print("Inference complete.")
    return df_res

# Run
df_results = run_sensitivity_analysis()

Analysis

In [12]:
# ==============================================================================
# 4. METRICS & REPORT
# ==============================================================================

if df_results is not None and not df_results.empty:
    print("\n" + "="*60)
    print(" SENSITIVITY ANALYSIS REPORT")
    print("="*60)

    grouped = df_results.groupby(['Ikey', 'UniProt'])
    
    # Metrics
    binding_stability = grouped.apply(lambda x: (x['Binding_Prob'] > 0.5).mean())
    activity_consistency = grouped.apply(lambda x: (x['Pred_Class'] == x['Pred_Class'].mode()[0]).mean())
    prediction_variance = grouped['Prob_Agonist'].std()
    
    print(f"\n[1] Binding Prediction Stability")
    print(f"    - Mean Retention Rate: {binding_stability.mean()*100:.2f}%")
    print(f"    (Percentage of conformers correctly maintained as binders)")

    print(f"\n[2] Functional Prediction Consistency")
    print(f"    - Mean Concordance Rate: {activity_consistency.mean()*100:.2f}%")
    print(f"    (Percentage of conformers agreeing with the majority vote)")

    print(f"\n[3] Prediction Uncertainty")
    print(f"    - Mean Std Dev (Agonist Prob): {prediction_variance.mean():.4f}")
    print(f"    (Low variance implies robustness to input conformation)")

    # Optional: Save detailed metrics
    metrics_df = pd.DataFrame({
        'Binding_Stability': binding_stability,
        'Activity_Consistency': activity_consistency,
        'Agonist_Prob_Std': prediction_variance
    })
    metrics_df.to_csv(CONFIG['OUTPUT_DIR'] / "robustness_metrics.csv")
    print(f"\nDetailed metrics saved to {CONFIG['OUTPUT_DIR']}")


  ANALYSIS RESULT: ROBUSTNESS & ENSEMBLE GAIN

[1] Binding Stability (Stage 1)
   - Mean Retention Rate: 99.91%
   - Interpretation: On average, 99.9% of conformers are classified as binders.

[2] Activity Consistency (Model Robustness)
   - Mean Concordance Rate: 99.81%
   - Interpretation: Across conformers, the model predicts the same class 99.8% of the time.

[3] Prediction Uncertainty (Variance)
   - Mean Standard Deviation: 0.0035
   - Interpretation: Lower variance indicates that the model is less sensitive to conformational variation.

[4] Performance Comparison (Ensemble Gain)
   - Baseline (Single) BAcc: 0.8059 (Acc: 0.8235)
   - Ensemble (Avg)    BAcc: 0.8079 (Acc: 0.8250)
   - Improvement:       +0.20%p

Summary metrics saved to ./revision/analysis_results/ETKDG/robustness_metrics_summary.csv
